# Simple PID loop double check
Craig Lage - 20Mar26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt
from lsst.ts.ofc.state_estimator import StateEstimator
from lsst.ts.ofc.ofc import OFC
from lsst.ts.ofc import OFCData


## Get the Nightly report data and the PID logs.

In [ ]:
day_obs = 20260404

# Load the nightly table.
# The parquet file was created with: 
# https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb
parquet_file = f"/home/cslage/MTAOS/times_square_reports/nightly_aos_table_{day_obs}.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/cslage/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)
    

In [ ]:
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]

vmode_labels = ['Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry', 
                'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
                'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
                'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
                'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
                'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical']


## Next, we verify that the corrections logged are what is actually being applied
## Basically dof_state(n+1) = - correction(n) + dof_state(n)

In [ ]:
# Finally, this checks - Key progress!!
indices = list(range(0, 17)) + list(range(30,35))
expId1 = 2026032900273

correction = pidDict[expId1]['Correction'] # From the "Correction" in the log message.
print(f"Correction {expId1}")
print(correction)
print()
expId2 = expId1 + 1
seqNum1 = int(expId1 - 1E5 * day_obs)
seq_table1 = table[table['seq'] == seqNum1]
dof_state1 = seq_table1['dof_state'].values[0]
seqNum2 = int(expId2 - 1E5 * day_obs)
seq_table2 = table[table['seq'] == seqNum2]
dof_state2 = seq_table2['dof_state'].values[0]

print(f"dof_state_{seqNum2}")
print(dof_state2[indices])
print()
print(f"dof_state_{seqNum1} - Correction")
print(dof_state1[indices] - correction)
print("These are identical within a small error")

## Next I try to calculate the correction.  These now agree if I use the logged
## "Error" term as dof_visit.

In [ ]:
seqNum = 327
expId = int(1E5 * day_obs + seqNum)
dof_visit = pidDict[expId]['Error']  # From the "Error before any gains..." in the log message.
print(f"dof_visit seqNum{seqNum}")
print(dof_visit)
print()
correction = pidDict[expId]['Correction'] # From the "Correction" in the log message.
print(f"Correction {expId}")
print(correction)
print()
Kp = pidDict[expId]['Kp']
Ki = pidDict[expId]['Ki']
clip_i = pidDict[expId]['Clipped_I'] # From the "Clipped_I" in the log message.
calculated_correction = dof_visit * Kp + clip_i * Ki
print(f"Calculated correction {expId}")
print(calculated_correction)
print("These are identical within a small error")

## Verifying agreement for the whole night.

In [ ]:
corrections = []
calculated_corrections = []
for expId in pidDict.keys():
    thisDict = pidDict[expId]
    correction = pidDict[expId]['Correction'] # The Correction term in the log
    
    dof_visit = pidDict[expId]['Error'] # The error term in the log
    clip_i = pidDict[expId]['Clipped_I'] # The Clipped_I term in the log
    Kp = pidDict[expId]['Kp']
    Ki = pidDict[expId]['Ki']
    if len(dof_visit) != len(Kp):
        print(f"{expId} shapes don't match.")
        continue
        
    calculated_correction = dof_visit * Kp + clip_i * Ki
    calculated_corrections.append(calculated_correction)
    corrections.append(correction)
        
corrections = np.array(corrections)
calculated_corrections = np.array(calculated_corrections)

indices = list(range(0, 17)) + list(range(30,35)) # This just collapses the 50 DOF down to the 22 we are using
fig, axes = plt.subplots(5, 5, figsize=(15,15))
plt.subplots_adjust(hspace=0.5, wspace = 0.6)
plt.suptitle(f"Logged corrections vs calculated corrections {day_obs}", fontsize=18, y=0.95)
for i in range(5):
    for j in range(5):
        index = i + 5 * j
        if index > 21:
            axes[i][j].set_axis_off()
            continue
        label = all_labels[indices[index]]
        axes[i][j].scatter(corrections[:,index], calculated_corrections[:,index])
        axes[i][j].set_title(label)
        axes[i][j].set_xlabel("Logged correction")
        axes[i][j].set_ylabel("Calculated correction")
        
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/PID_Corrections_Check_{day_obs}.png", 
            bbox_inches='tight', pad_inches=0.1)

## Now attempt to calculate the Error term from the Zernike deviations
## I've tried multiple ways to try to get it to agree, so far without success.

In [ ]:
ofcData = OFCData(name="lsst")
ofcData.configure_controller()
await ofcData.configure_instrument("lsst")
ofc = OFC(ofcData)
# Restrict to standard_22 DOFs (5+5+7+5 = 22)
m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:7] = True
m2_bending[:5] = True
ofcData.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)
ofc.controller.kp = 0.3
ofc.controller.ki = 0
ofc.controller.kd = 0
ofcData.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
#ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

In [ ]:
np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]).shape

In [ ]:
ofc.calculate_corrections?

In [ ]:
expId = 2026040400143
sourceExpId = pidDict[expId]['Source']
sourceSeqNum = int(sourceExpId - 1E5 * day_obs)
this_table = table[table['seq'] == sourceSeqNum]
print(f"ExpId = {expId}, Source ExpId = {sourceExpId}")
zernikes = np.zeros([4,23])
zernikes[0,:] = this_table['zk_deviation_R00'].values[0]
zernikes[1,:] = this_table['zk_deviation_R04'].values[0]
zernikes[2,:] = this_table['zk_deviation_R40'].values[0]
zernikes[3,:] = this_table['zk_deviation_R44'].values[0]
print("Zernikes")
print(zernikes)
print()
sensor_ids = [191, 195, 199, 203]
rotation_angle = 0.0
filter_name = 'I'

calculated_corrections = ofc.calculate_corrections(zernikes, sensor_ids, filter_name,
                                                   rotation_angle, subtract_intrinsics=False,
                                                  control_vmodes=False)
print()
error_term = pidDict[expId]['Error']
print("Logged error")
print(error_term)

print()
logged_corrections = pidDict[expId]['Correction']
print("Logged correction")
print(logged_corrections)


In [ ]:
array([-0.22881990671157837, -0.12116247415542603, 0.37163230776786804,
       -0.1192484200000763, -0.055395904928445816, 0.12014810740947723,
       -0.1532166451215744, 0.0508444644510746, 0.16408181190490723,
       -0.16008661687374115, -0.1021241769194603, -0.13293443620204926,
       -0.08718220144510269, 0.08307468146085739, -0.02233097329735756,
       -0.004530721344053745, -0.009979652240872383, 0.04017117992043495,
       0.007906108163297176, -0.005345472600311041, 0.06264752149581909,
       None, None, None, None], dtype=object)

In [ ]:
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId
import lsst.summit.utils.butlerUtils as butlerUtils
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
expId = 2026040400010
dataId = {'exposure':expId, 'instrument':'LSSTCam'}
firstExpRecord = getExpRecordFromDataId(butler, dataId)
expId = 2026040400600
dataId = {'exposure':expId, 'instrument':'LSSTCam'}
lastExpRecord = getExpRecordFromDataId(butler, dataId)

In [ ]:
wavefront_errors =getEfdData(client, 
    topic="lsst.sal.MTAOS.logevent_wavefrontError",
    columns=[
        f"nollZernikeValues{i}" for i in range(27)
    ] + [
        f"nollZernikeIndices{i}" for i in range(27)
    ] + [
        "sensorId",
        "visitId"
    ],
    begin=firstExpRecord.timespan.begin,
    end=lastExpRecord.timespan.end                         
)
len(wavefront_errors)

In [ ]:
these_wavefront_errors = wavefront_errors[wavefront_errors['visitId']==2026040400146]

In [ ]:
test = these_wavefront_errors[these_wavefront_errors['sensorId']==191].values[0][0:21]
test

In [ ]:
zernikes[0]

In [ ]:
this_table['zk_deviation_R00'].values

In [ ]:
expId = 2026040400146
dataId = {'visit':expId, 'instrument':'LSSTCam', 'detector':191}
z_191 = butler.get('zernikes', dataId=dataId)

In [ ]:
z_191

In [ ]:
z_191.columns

In [ ]:
z_191['Z4_deviation'][0]

In [ ]:
z_191['Z6_deviation'][0]

In [ ]:
(194.14 + 150.50 + 279.52 + 242.08 + 306.37 + 302.62) / 6.0

In [ ]:
expId = 2026040400364
these_wavefront_errors = wavefront_errors[wavefront_errors['visitId']==2026040400362]
sourceExpId = pidDict[expId]['Source']
sourceSeqNum = int(sourceExpId - 1E5 * day_obs)
this_table = table[table['seq'] == sourceSeqNum]
print(f"ExpId = {expId}, Source ExpId = {sourceExpId}")
zernikes = np.zeros([4,25])
zernikes[0,0:21] = these_wavefront_errors[these_wavefront_errors['sensorId']==191].values[0][0:21]
zernikes[1,0:21] = these_wavefront_errors[these_wavefront_errors['sensorId']==195].values[0][0:21]
zernikes[2,0:21] = these_wavefront_errors[these_wavefront_errors['sensorId']==199].values[0][0:21]
zernikes[3,0:21] = these_wavefront_errors[these_wavefront_errors['sensorId']==203].values[0][0:21]
print("Zernikes")
print(zernikes)
print()
sensor_ids = [191, 195, 199, 203]
rotation_angle = 0.0
filter_name = 'I'

calculated_corrections = ofc.calculate_corrections(zernikes, sensor_ids, filter_name,
                                                   rotation_angle, subtract_intrinsics=True,
                                                  control_vmodes=False)
print()
error_term = pidDict[expId]['Error']
print("Logged error")
print(error_term)

print()
logged_corrections = pidDict[expId]['Correction']
print("Logged correction")
print(logged_corrections)


In [ ]:
a = "-3.45100956e+01  6.89927849e-04  2.22809155e-02 -6.33015212e-04\
 -9.84891917e-04  3.81290099e+01 -7.77367451e-05 -2.65749619e-03\
 -5.84314822e-04  6.07756550e-04 -1.07418285e-02 -1.67113407e-02\
  1.01720818e-01  8.59468688e-03  9.60896613e-03  5.38351761e-03\
  3.70503736e-02  2.04294498e-02  2.87208409e-02  1.27419132e-02\
 -5.02919276e-03  6.91879032e-02"
test2 = [float(x) for x in a.split()]

In [ ]:
test2 / error_term